In [ ]:
import os
import subprocess

from kaggle_secrets import UserSecretsClient

REPO = "small-model-alignment"
GITHUB_USER = "ahmedamirkol01-art"
os.chdir("/kaggle/working")
if not os.path.exists(REPO):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    result = subprocess.run(["git", "clone", f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git"], capture_output=True, text=True)
    print("cloned" if result.returncode == 0 else result.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://github.com/{GITHUB_USER}/{REPO}.git"])
os.chdir(f"/kaggle/working/{REPO}")
os.environ["PYTHONPATH"] = "."
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!git log --oneline -3

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("visible GPUs:", torch.cuda.device_count())

In [ ]:
import glob

SFT = glob.glob("/kaggle/input/**/outputs/sft/best", recursive=True)[0]
DPO_HH = glob.glob("/kaggle/input/**/outputs/dpo_hh/best", recursive=True)[0]
DPO_UF = glob.glob("/kaggle/input/**/outputs/dpo_ultrafeedback/best", recursive=True)[0]
os.environ.update(SFT=SFT, DPO_HH=DPO_HH, DPO_UF=DPO_UF)
print(SFT, DPO_HH, DPO_UF, sep="\n")

In [ ]:
!python scripts/generate.py --model_path {SFT} --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-sft --output_path outputs/sanity/sft_alpaca.json --limit 5
!python scripts/generate.py --model_path {DPO_UF} --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-dpo-uf --output_path outputs/sanity/dpo_uf_alpaca.json --limit 5

In [ ]:
%%bash
A=eval_sets/alpaca_eval_300_seed42.json
H=eval_sets/hh_rlhf_eval_300_seed42.json
G=outputs/generations
(
CUDA_VISIBLE_DEVICES=0 python scripts/generate.py --model_path $SFT --eval_set $A --generator qwen2.5-0.5b-sft --output_path $G/alpaca_sft.json
CUDA_VISIBLE_DEVICES=0 python scripts/generate.py --model_path $DPO_UF --eval_set $A --generator qwen2.5-0.5b-dpo-uf --output_path $G/alpaca_dpo_uf.json
CUDA_VISIBLE_DEVICES=0 python scripts/generate.py --model_path $DPO_HH --eval_set $A --generator qwen2.5-0.5b-dpo-hh --output_path $G/alpaca_dpo_hh.json
CUDA_VISIBLE_DEVICES=0 python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_set $A --generator qwen2.5-0.5b-base --output_path $G/alpaca_base.json
) > outputs/gen_gpu0.log 2>&1 &
(
CUDA_VISIBLE_DEVICES=1 python scripts/generate.py --model_path $SFT --eval_set $H --generator qwen2.5-0.5b-sft --output_path $G/hh_sft.json
CUDA_VISIBLE_DEVICES=1 python scripts/generate.py --model_path $DPO_HH --eval_set $H --generator qwen2.5-0.5b-dpo-hh --output_path $G/hh_dpo_hh.json
CUDA_VISIBLE_DEVICES=1 python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_set $H --generator qwen2.5-0.5b-base --output_path $G/hh_base.json
) > outputs/gen_gpu1.log 2>&1 &
wait
grep -h -E "saved|Error" outputs/gen_gpu0.log outputs/gen_gpu1.log

In [ ]:
!python scripts/build_pairs.py --candidate_path outputs/generations/hh_dpo_hh.json --baseline_path outputs/generations/hh_sft.json --output_path outputs/eval/hh_dpo_vs_sft_pairs.jsonl
!python scripts/build_pairs.py --candidate_path outputs/generations/alpaca_dpo_uf.json --baseline_path outputs/generations/alpaca_sft.json --output_path outputs/eval/alpaca_dpo_uf_vs_sft_pairs.jsonl
!python scripts/build_pairs.py --candidate_path outputs/generations/alpaca_dpo_hh.json --baseline_path outputs/generations/alpaca_sft.json --output_path outputs/eval/alpaca_dpo_hh_vs_sft_pairs.jsonl

In [ ]:
!cd /kaggle/working && zip -r -q generation_outputs.zip small-model-alignment/outputs
!ls -lh outputs/generations outputs/eval